In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt
from scipy.interpolate import UnivariateSpline
import seaborn as sns
from iminuit import cost, Minuit 
import numpy as np
import pickle

sns.set_context('talk', font_scale=1.0)
sns.set_palette('colorblind')

from depth_helpers import *

plot_dir = '/Users/Sophie/Desktop/COSI/calibrations/Data/trapping_figures/'
sim_dCCE_path = '/Users/Sophie/Desktop/COSI/calibrations/Data/SimulatedCSVs/HP41520_1/HP415201_dCCE_1000cm.csv'

sim_dCCE = load_sim_dCCE(sim_dCCE_path)

def e_depth_plot(z, ae, b, c):
    CCE = ae*(1.-b*sim_dCCE[1][::-1])*(1.-c*sim_dCCE[2][::-1])
    return UnivariateSpline(sim_dCCE[0], CCE)(z)

def h_depth_plot(z, ah, b, c):
    CCE = ah*(1.-b*sim_dCCE[3][::-1])*(1.-c*sim_dCCE[4][::-1])
    return UnivariateSpline(sim_dCCE[0], CCE)(z)

# Load the experimental depth plots

Written by `depth_analysis_preLL2025_SEH.ipynb` and `depth_analysis_postLL2025_SEH.ipynb` (`depth_plot_<source>allACDC_<epoch>.txt`, pre-LL in `plot_dir`, post-LL next to this notebook). Each holds the depth bin edges and the HV (`e`) and LV (`h`) photopeak centroids / line energy with errors.

In [ ]:
sources = ['Cs137', 'Am241']
epoch_list = ['preLL', 'postLL1']
label_list = ['Pre-LL', 'Post-LL']
### Depth bins at each edge excluded from the CCE fit (same as the analysis notebooks), drawn as 'x'.
trim_dict = {'Cs137': 1, 'Am241': 0}
### Where each analysis notebook writes its depth_plot_*.txt files
depth_plot_dir = {'preLL': plot_dir, 'postLL1': './'}

depth_plot_dict = {}
for source in sources:
    for epoch in epoch_list:
        with open(depth_plot_dir[epoch] + 'depth_plot_' + source + 'allACDC_' + epoch + '.txt', 'rb') as file:
            depth_plot_dict[source, epoch] = pickle.load(file)

# Fit the simulated CCE curve for each epoch

Same fits as the analysis notebooks: Cs137 constrains ae, ah and the trapping parameters b (electrons) and c (holes); Am241 only fits its own ae, ah with b, c fixed to the Cs137 values of the same epoch.

In [ ]:
def fit_mask(source, n):
    mask = np.zeros(n, dtype=bool)
    mask[trim_dict[source]:n-trim_dict[source]] = True
    return mask

def fit_depth_plot(source, epoch, fixed_bc=None):
    z_bins, e_trapping, h_trapping = depth_plot_dict[source, epoch]
    z_list = (z_bins[:-1] + z_bins[1:])/2.
    mask = fit_mask(source, len(z_list))
    e_ok = mask & np.isfinite(e_trapping[0]) & (e_trapping[1] > 0)
    h_ok = mask & np.isfinite(h_trapping[0]) & (h_trapping[1] > 0)
    c = cost.LeastSquares(z_list[e_ok], e_trapping[0][e_ok], e_trapping[1][e_ok], e_depth_plot) + \
        cost.LeastSquares(z_list[h_ok], h_trapping[0][h_ok], h_trapping[1][h_ok], h_depth_plot)
    if fixed_bc is None:
        m = Minuit(c, ae=np.nanmax(e_trapping[0]), ah=np.nanmax(h_trapping[0]), b=1.0, c=9.)
        m.limits['b', 'c'] = (0, None)
    else:
        m = Minuit(c, ae=np.nanmax(e_trapping[0]), ah=np.nanmax(h_trapping[0]), b=fixed_bc[0], c=fixed_bc[1])
        m.fixed['b', 'c'] = True
    m.migrad()
    m.hesse()
    return m

param_dict = {}
for epoch in epoch_list:
    param_dict['Cs137', epoch] = fit_depth_plot('Cs137', epoch)
    param_dict['Am241', epoch] = fit_depth_plot('Am241', epoch, fixed_bc=param_dict['Cs137', epoch].values['b', 'c'])

### Summary table, also saved next to the depth plots
lines = ['# source epoch ae ae_err ah ah_err b b_err c c_err chi2/ndof']
for (source, epoch), m in param_dict.items():
    v = m.values
    e = {p: (0. if m.fixed[p] else m.errors[p]) for p in ['ae', 'ah', 'b', 'c']}   # b, c are fixed for Am241
    lines.append('%s %s %.6f %.6f %.6f %.6f %.4f %.4f %.4f %.4f %.2f' % (source, epoch, v['ae'], e['ae'], v['ah'], e['ah'], v['b'], e['b'], v['c'], e['c'], m.fmin.reduced_chi2))
print('\n'.join(lines))
with open('CCE_fit_params_2025.txt', 'w') as file:
    file.write('\n'.join(lines) + '\n')

# Centroid energy vs. depth

In [ ]:
for source in sources:
    line_e = source_dict[source]
    for k, carrier, axlabel in [(1, 'HV', 'etrapping'), (2, 'LV', 'htrapping')]:
        plt.figure()
        for i, epoch in enumerate(epoch_list):
            z_bins = depth_plot_dict[source, epoch][0]
            trapping = depth_plot_dict[source, epoch][k]
            z_list = (z_bins[:-1] + z_bins[1:])/2.
            z_err = (z_bins[1:]-z_bins[:-1])/2.
            plt.errorbar(z_list, trapping[0]*line_e, xerr=z_err, yerr=trapping[1]*line_e, fmt=".", color='C'+str(i+2), label=label_list[i])
        plt.legend(fontsize=12)
        plt.axhline(line_e, ls='--', color='red')
        plt.title(source + ', ' + carrier + ' signal', fontsize='small')
        plt.xlabel("Detector Depth (cm)"); plt.ylabel("Centroid Energy (keV)")
        plt.tight_layout()
        plt.savefig(plot_dir + axlabel + '_' + source + '_' + str(len(epoch_list)) + 'epochs.pdf')
        plt.show()
        plt.close()

# Normalized CCE vs. depth with the fitted simulation curves

In [ ]:
for source in sources:
    for k, carrier, norm_par, curve, color in [(1, 'HV', 'ae', e_depth_plot, 'C0'), (2, 'LV', 'ah', h_depth_plot, 'C1')]:
        plt.figure()
        for i, epoch in enumerate(epoch_list):
            m = param_dict[source, epoch]
            z_bins = depth_plot_dict[source, epoch][0]
            trapping = depth_plot_dict[source, epoch][k]
            z_list = (z_bins[:-1] + z_bins[1:])/2.
            z_err = (z_bins[1:]-z_bins[:-1])/2.
            norm = m.values[norm_par]
            mask = fit_mask(source, len(z_list))
            plt.errorbar(z_list[mask], (trapping[0]/norm)[mask], xerr=z_err[mask], yerr=(trapping[1]/norm)[mask], fmt=".", color='C'+str(i+2), label=label_list[i])
            if (~mask).any():
                plt.errorbar(z_list[~mask], (trapping[0]/norm)[~mask], xerr=z_err[~mask], yerr=(trapping[1]/norm)[~mask], fmt="x", color='C'+str(i+2))
            plt.plot(sim_dCCE[0], curve(sim_dCCE[0], 1.0, *m.values['b', 'c']), color='C'+str(i+2), lw=0.7)
        plt.legend(fontsize=12)
        plt.title(source, fontsize='small')
        plt.xlabel("Detector Depth (cm)"); plt.ylabel(carrier + " CCE", color=color)
        plt.tight_layout()
        plt.savefig(plot_dir + carrier + '_CCE_' + source + '_' + str(len(epoch_list)) + 'epochs.pdf')
        plt.show()
        plt.close()

# Trapping products by epoch (from the Cs137 fits)

In [ ]:
x = np.arange(len(epoch_list))
b_list = np.array([param_dict['Cs137', epoch].values['b'] for epoch in epoch_list])
b_err = np.array([param_dict['Cs137', epoch].errors['b'] for epoch in epoch_list])
c_list = np.array([param_dict['Cs137', epoch].values['c'] for epoch in epoch_list])
c_err = np.array([param_dict['Cs137', epoch].errors['c'] for epoch in epoch_list])

plt.figure()
plt.errorbar(x, b_list, yerr=b_err, marker='.', label='Electron trapping')
plt.errorbar(x, c_list, yerr=c_err, marker='.', label='Hole trapping')
plt.xticks(x, label_list)
plt.legend()
plt.ylabel(r'$[n\sigma]/\mathrm{(1000\,cm})$')
plt.tight_layout()
plt.savefig(plot_dir + 'trapping_products_Cs137_' + str(len(epoch_list)) + 'epochs.pdf')
plt.show()
plt.close()